In [1]:
# ─────────────────────────────────────────────────────────────
# Cell 1 · Install all required packages
# ─────────────────────────────────────────────────────────────
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    else:
        print(f'✅  {cmd[:80]}')

run('pip install -q ultralytics==8.3.40')
run('pip install -q supervision==0.25.1')
run('pip install -q filterpy==1.4.5')
run('pip install -q scipy numpy opencv-python-headless')

# lap is required by ByteTrack inside supervision
run('pip install -q lapx')

print('\n🎉 All packages installed.')

✅  pip install -q ultralytics==8.3.40
✅  pip install -q supervision==0.25.1
✅  pip install -q filterpy==1.4.5
✅  pip install -q scipy numpy opencv-python-headless
✅  pip install -q lapx

🎉 All packages installed.


In [2]:
# ─────────────────────────────────────────────────────────────
# Cell 2 · Import all libraries
# ─────────────────────────────────────────────────────────────
import cv2
import numpy as np
import os
import glob
import math
import warnings
warnings.filterwarnings('ignore')

from scipy.signal import savgol_filter
from scipy.interpolate import UnivariateSpline


from filterpy.kalman import KalmanFilter

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from ultralytics import YOLO
import supervision as sv

from google.colab import files

print('✅  All libraries imported successfully.')
print(f'   OpenCV  : {cv2.__version__}')
print(f'   NumPy   : {np.__version__}')
print(f'   sv      : {sv.__version__}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅  All libraries imported successfully.
   OpenCV  : 4.13.0
   NumPy   : 2.0.2
   sv      : 0.25.1


In [3]:
# ─────────────────────────────────────────────────────────────
# Cell 3 · Upload video from local computer
# ─────────────────────────────────────────────────────────────
print('📂  Please select your cricket video file …')
uploaded = files.upload()

# Automatically detect the uploaded filename
if not uploaded:
    raise RuntimeError('No file uploaded. Please re-run this cell and upload a video.')

VIDEO_PATH = list(uploaded.keys())[0]
print(f'\n✅  Uploaded : {VIDEO_PATH}')
print(f'   Size      : {os.path.getsize(VIDEO_PATH) / 1e6:.2f} MB')

📂  Please select your cricket video file …


Saving cricket clips 10.mp4 to cricket clips 10.mp4

✅  Uploaded : cricket clips 10.mp4
   Size      : 2.33 MB


In [4]:
# ─────────────────────────────────────────────────────────────
# Cell 4 · Read video metadata
# ─────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f'Cannot open video: {VIDEO_PATH}')

FPS          = cap.get(cv2.CAP_PROP_FPS)
FRAME_WIDTH  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
FRAME_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

cap.release()

# Clamp FPS to a sane range
if FPS <= 0 or FPS > 240:
    FPS = 30.0
    print('⚠️  FPS unreadable, defaulting to 30.')

print('📹  Video Information')
print(f'   Path         : {VIDEO_PATH}')
print(f'   Resolution   : {FRAME_WIDTH} × {FRAME_HEIGHT}')
print(f'   FPS          : {FPS:.2f}')
print(f'   Total Frames : {TOTAL_FRAMES}')
print(f'   Duration     : {TOTAL_FRAMES / FPS:.2f} s')

📹  Video Information
   Path         : cricket clips 10.mp4
   Resolution   : 1920 × 1080
   FPS          : 30.42
   Total Frames : 73
   Duration     : 2.40 s


In [8]:
# ─────────────────────────────────────────────────────────────
# Cell 5 · Load custom cricket-ball model
# ─────────────────────────────────────────────────────────────

from ultralytics import YOLO
from google.colab import files

uploaded = files.upload()

# Path to your trained weights
MODEL_PATH = "best .pt"

# Load model
model = YOLO(MODEL_PATH)

# Detection thresholds
CONF_THRESHOLD = 0.20
IOU_THRESHOLD = 0.45

print("✅ Custom model loaded successfully!")
print(f"Model path: {MODEL_PATH}")
print(f"Confidence threshold: {CONF_THRESHOLD}")
print(f"IoU threshold: {IOU_THRESHOLD}")

Saving best .pt to best  (2).pt
✅ Custom model loaded successfully!
Model path: best .pt
Confidence threshold: 0.2
IoU threshold: 0.45


In [9]:
# ─────────────────────────────────────────────────────────────
# Cell 6 · Frame-by-frame Ball Detection
# ─────────────────────────────────────────────────────────────

raw_detections = []
debug_frames = []

cap = cv2.VideoCapture(VIDEO_PATH)

frame_idx = 0
found_count = 0

print("🔍 Running Ball Detection...")

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model(
        frame,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        verbose=False
    )

    det = None

    if len(results[0].boxes) > 0:

        # choose highest confidence detection
        boxes = results[0].boxes

        best_idx = boxes.conf.argmax()

        box = boxes[best_idx]

        conf = float(box.conf[0])

        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2

        radius = min(x2 - x1, y2 - y1) / 2

        det = (cx, cy, radius, conf)

    if det is not None:

        cx, cy, radius, conf = det

        raw_detections.append((frame_idx, cx, cy, radius))

        found_count += 1

        if len(debug_frames) < 5:

            dbg = frame.copy()

            cv2.circle(
                dbg,
                (int(cx), int(cy)),
                max(int(radius), 5),
                (0,0,255),
                2
            )

            cv2.putText(
                dbg,
                f"{conf:.2f}",
                (int(cx)+10, int(cy)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0,0,255),
                2
            )

            debug_frames.append((frame_idx, dbg))

    else:
        raw_detections.append(None)

    frame_idx += 1

    if frame_idx % 50 == 0:

        print(
            f"Frame {frame_idx}/{TOTAL_FRAMES} | detections = {found_count}"
        )

cap.release()

ACTUAL_FRAMES = frame_idx

print()
print("✅ Detection complete")
print("Frames processed:", ACTUAL_FRAMES)
print("Frames with ball:", found_count)

🔍 Running Ball Detection...
Frame 50/73 | detections = 43

✅ Detection complete
Frames processed: 72
Frames with ball: 65


In [10]:
# ─────────────────────────────────────────────────────────────
# Cell 7 · Kalman Ball Tracker — 3D (x, y, z)
# z is estimated from inverse of ball radius (depth proxy)
# Larger apparent radius → ball is closer → smaller Z
# ─────────────────────────────────────────────────────────────

from filterpy.kalman import KalmanFilter

REFERENCE_RADIUS = 10.0  # baseline radius at "standard" distance

# State: [x, y, z, vx, vy, vz]   Measurement: [x, y, z]
kf = KalmanFilter(dim_x=6, dim_z=3)

dt = 1

kf.F = np.array([
    [1,0,0,dt,0,0],
    [0,1,0,0,dt,0],
    [0,0,1,0,0,dt],
    [0,0,0,1,0,0],
    [0,0,0,0,1,0],
    [0,0,0,0,0,1],
], dtype=float)

kf.H = np.array([
    [1,0,0,0,0,0],
    [0,1,0,0,0,0],
    [0,0,1,0,0,0],
], dtype=float)

kf.R = np.eye(3) * 10
kf.Q = np.eye(6) * 0.1
kf.P = np.eye(6) * 100

track_lookup = {}
initialized  = False

print("🔁 Running 3D Kalman tracking...")

for det in raw_detections:

    if det is None:
        if initialized:
            kf.predict()
        continue

    frame_idx, cx, cy, radius = det

    # Depth proxy: smaller radius → ball is farther → larger Z
    z = REFERENCE_RADIUS / max(radius, 0.5)

    if not initialized:
        kf.x = np.array([[cx],[cy],[z],[0],[0],[0]])
        initialized = True

    kf.predict()
    kf.update(np.array([[cx],[cy],[z]]))

    x = float(kf.x[0])
    y = float(kf.x[1])
    z = float(kf.x[2])

    track_lookup[frame_idx] = (x, y, z, radius)

print()
print("✅ 3D Kalman tracking complete")
print("Tracked frames:", len(track_lookup))


🔁 Running 3D Kalman tracking...

✅ 3D Kalman tracking complete
Tracked frames: 65


In [11]:
# ─────────────────────────────────────────────────────────────
# Remove abnormal jumps  (3D-aware)
# ─────────────────────────────────────────────────────────────

clean_track = {}

prev_x = None
prev_y = None
prev_z = None

MAX_DIST = 100   # pixel threshold for X/Y jump

for frame_idx in sorted(track_lookup.keys()):

    x, y, z, r = track_lookup[frame_idx]

    if prev_x is not None:

        dist = np.sqrt(
            (x - prev_x)**2 +
            (y - prev_y)**2
        )

        if dist > MAX_DIST:
            continue

    clean_track[frame_idx] = (x, y, z, r)

    prev_x, prev_y, prev_z = x, y, z

track_lookup = clean_track

print("✅ Outliers removed")
print("Remaining points:", len(track_lookup))


✅ Outliers removed
Remaining points: 27


In [12]:
# ─────────────────────────────────────────────────────────────
# Cell 8 · Trajectory smoothing — aggressive for clean DRS tube
# ─────────────────────────────────────────────────────────────

from scipy.signal import savgol_filter
from scipy.interpolate import UnivariateSpline
import numpy as np

frame_indices = sorted(track_lookup.keys())

if len(frame_indices) < 5:
    raise RuntimeError("Too few ball detections!")

cx_arr = np.array([track_lookup[i][0] for i in frame_indices], dtype=float)
cy_arr = np.array([track_lookup[i][1] for i in frame_indices], dtype=float)
cz_arr = np.array([track_lookup[i][2] for i in frame_indices], dtype=float)
r_arr  = np.array([track_lookup[i][3] for i in frame_indices], dtype=float)

n = len(cx_arr)

# ── Step 1: Heavy Savitzky-Golay pass ────────────────────────
# Use a large window (up to 51 frames) to kill high-freq noise
win = min(51, n)
if win % 2 == 0: win -= 1
if win < 5: win = 5

poly = 2   # low poly order → smoother, less wiggle

sg_x = savgol_filter(cx_arr, window_length=win, polyorder=poly)
sg_y = savgol_filter(cy_arr, window_length=win, polyorder=poly)
sg_z = savgol_filter(cz_arr, window_length=win, polyorder=poly)

# ── Step 2: Spline fit on the SG-filtered data ───────────────
# s (smoothing factor) controls how tightly spline fits the data
# Higher s = smoother curve; tune between 0.1*n and 2*n
t = np.arange(n, dtype=float)

s_factor = max(n * 0.5, 10)   # adjust: larger = smoother

try:
    spl_x = UnivariateSpline(t, sg_x, k=3, s=s_factor)
    spl_y = UnivariateSpline(t, sg_y, k=3, s=s_factor)
    spl_z = UnivariateSpline(t, sg_z, k=3, s=s_factor)

    t_fine = np.linspace(0, n-1, n)
    smooth_x = spl_x(t_fine)
    smooth_y = spl_y(t_fine)
    smooth_z = spl_z(t_fine)

except Exception as e:
    print(f"⚠️  Spline fallback (SG only): {e}")
    smooth_x = sg_x
    smooth_y = sg_y
    smooth_z = sg_z

smooth_r = savgol_filter(r_arr, window_length=win, polyorder=2)

print("✅ 3D Trajectory smoothing complete")
print(f"   Points  : {len(smooth_x)}")
print(f"   SG win  : {win}  poly: {poly}  spline s: {s_factor:.1f}")

# Preview
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(cx_arr,   alpha=0.4, label="raw X");  axes[0].plot(smooth_x, label="smooth X"); axes[0].legend(); axes[0].set_title("X")
axes[1].plot(cy_arr,   alpha=0.4, label="raw Y");  axes[1].plot(smooth_y, label="smooth Y"); axes[1].legend(); axes[1].set_title("Y")
axes[2].plot(cz_arr,   alpha=0.4, label="raw Z");  axes[2].plot(smooth_z, label="smooth Z"); axes[2].legend(); axes[2].set_title("Z depth")
plt.tight_layout(); plt.show()


✅ 3D Trajectory smoothing complete
   Points  : 27
   SG win  : 27  poly: 2  spline s: 13.5


In [13]:

# ─────────────────────────────────────────────────────────────
# Cell 9 · Hawk-Eye 3D tube renderer
# Broadcast-quality DRS glass tube — cylindrical volume render
# ─────────────────────────────────────────────────────────────

import cv2, math
import numpy as np
from scipy.ndimage import gaussian_filter

FOCAL_LENGTH = 10000.0
DEPTH_SCALE = 2.0


def _perspective_scale(z):
    z_safe = max(float(z), 0.1)
    return FOCAL_LENGTH / (FOCAL_LENGTH + z_safe * DEPTH_SCALE)


def _interpolate_segment(p1, p2, r1, r2, steps=12):
    """Return `steps` interpolated (x, y, r) sub-points along p1→p2."""
    pts = []
    for k in range(steps + 1):
        t = k / steps
        x = p1[0] + t * (p2[0] - p1[0])
        y = p1[1] + t * (p2[1] - p1[1])
        r = r1   + t * (r2   - r1)
        pts.append((x, y, r))
    return pts


def _draw_segment_tube(frame, p1, p2, r1, r2):

    x1, y1 = float(p1[0]), float(p1[1])
    x2, y2 = float(p2[0]), float(p2[1])

    dx = x2 - x1
    dy = y2 - y1

    seg_len = math.hypot(dx, dy)

    if seg_len < 1:
        return

    ux = dx / seg_len
    uy = dy / seg_len

    angle = np.degrees(np.arctan2(dy, dx))

    # Dense sampling
    steps = max(int(seg_len * 4), 40)

    overlay = frame.copy()

    for k in range(steps + 1):

        t = k / steps

        cx = x1 + t * dx
        cy = y1 + t * dy

        r = r1 + t * (r2 - r1)

        major_axis = int(r)
        minor_axis = int(r * 0.55)

        center = (int(cx), int(cy))

        #
        # Dark outer shell
        #
        cv2.ellipse(
            overlay,
            center,
            (major_axis, minor_axis),
            angle,
            0,
            360,
           (160,80,0),
            -1,
            cv2.LINE_AA
        )

        #
        # Main transparent body
        #
        cv2.ellipse(
            overlay,
            center,
            (int(major_axis*0.82), int(minor_axis*0.82)),
            angle,
            0,
            360,
           (255,180,40),
            -1,
            cv2.LINE_AA
        )

        #
        # # Highlight strip (upper-left)
        # #
        # hx = cx - (-uy) * r * 0.35
        # hy = cy - ux * r * 0.35

        # cv2.ellipse(
        #     overlay,
        #     (int(hx), int(hy)),
        #     (
        #         max(int(major_axis*0.15),1),
        #         max(int(minor_axis*0.15),1)
        #     ),
        #     angle,
        #     0,
        #     360,
        #    (255,240,200),
        #     -1,
        #     cv2.LINE_AA
        # )

    #
    # Blend very softly
    #
    cv2.addWeighted(
        overlay,
        0.12,
        frame,
        0.88,
        0,
        frame
    )

def draw_3d_tube(frame, trajectory_3d, base_radius=18, glow=True):
    """
    trajectory_3d : list of (x, y, z)
    base_radius   : tube half-width in pixels at z~0
    """
    pts = np.asarray(trajectory_3d, dtype=float)
    if len(pts) < 2:
        return

    # Screen-space positions + perspective-scaled radii
    screen = [
        (p[0], p[1], base_radius * _perspective_scale(p[2]))
        for p in pts
    ]

    # ── Painter's algorithm: far segments first ───────────────
    order = sorted(
        range(1, len(screen)),
        key=lambda i: -(screen[i-1][2] + screen[i][2])
    )

    for i in order:
        x1, y1, r1 = screen[i-1]
        x2, y2, r2 = screen[i]
        _draw_segment_tube(frame, (x1, y1), (x2, y2), r1, r2)
    hx, hy, hr = screen[-1]

    hx = int(round(hx))
    hy = int(round(hy))
    head_r = max(int(round(hr)), 4)


    # Ball head
    cv2.circle(
        frame,
        (hx, hy),
        head_r,
        (160,80,0)
        ,
        -1,
        cv2.LINE_AA
    )

    cv2.circle(
        frame,
        (hx, hy),
        int(head_r*0.75),
        (255,180,40),
        -1,
        cv2.LINE_AA
    )

    cv2.circle(
        frame,
        (hx, hy),
        int(head_r*0.30),
        (255,255,255),
        -1,
        cv2.LINE_AA
    )

    # highlight
    cv2.circle(
        frame,
        (
            int(hx-head_r*0.3),
            int(hy-head_r*0.3)
        ),
        max(int(head_r*0.15),1),
        (255,255,255),
        -1,
        cv2.LINE_AA
    )






In [14]:
# ─────────────────────────────────────────────────────────────
# Cell 10 · Per-frame rendering logic  (3D trajectory)
# ─────────────────────────────────────────────────────────────

TUBE_RADIUS        = 8
TRAIL_MAX_FRAMES   = ACTUAL_FRAMES
SHOW_FRAME_COUNTER = True
OVERLAY_LOGO_TEXT  = ' '


def render_drs_frame(frame, frame_idx, smooth_x, smooth_y, smooth_z,
                     radius=12, trail_length=15):
    """
    Draw the growing DRS tube on a single frame using 3-D coordinates.
    """
    out = frame.copy()

    start_fi = 0
    end_fi   = frame_idx + 1

    traj = []

    for i, real_frame in enumerate(frame_indices):

        if real_frame < start_fi:
            continue

        if real_frame > frame_idx:
            break

        # Include Z depth for true 3-D rendering
        traj.append((smooth_x[i], smooth_y[i], smooth_z[i]))

    if len(traj) >= 2:
        draw_3d_tube(out, traj, base_radius=radius, glow=True)

    # ── Frame counter overlay ─────────────────────────────────
    if SHOW_FRAME_COUNTER:
        cv2.putText(
            out,
            f'Frame {frame_idx:04d}',
            (12, FRAME_HEIGHT - 18),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55,
            (200, 200, 200), 1, cv2.LINE_AA
        )

    # ── DRS watermark ─────────────────────────────────────────
    if OVERLAY_LOGO_TEXT:
        font_scale = max(0.5, FRAME_WIDTH / 1280 * 0.9)
        (tw, th), _ = cv2.getTextSize(OVERLAY_LOGO_TEXT,
                                       cv2.FONT_HERSHEY_DUPLEX,
                                       font_scale, 2)
        tx = FRAME_WIDTH - tw - 16
        ty = th + 12
        cv2.putText(out, OVERLAY_LOGO_TEXT, (tx, ty),
                    cv2.FONT_HERSHEY_DUPLEX, font_scale,
                    (0, 255, 255), 2, cv2.LINE_AA)

    return out


# ── Quick preview ─────────────────────────────────────────────
preview_fi = frame_indices[len(frame_indices) // 2]
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, preview_fi)
ret, sample_frame = cap.read()
cap.release()

if ret:
    preview = render_drs_frame(
        sample_frame,
        preview_fi,
        smooth_x, smooth_y, smooth_z,
        radius=TUBE_RADIUS,
        trail_length=TRAIL_MAX_FRAMES
    )
    plt.figure(figsize=(12, 6))
    plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
    plt.title(f'3D DRS preview — frame {preview_fi}')
    plt.axis('off')
    plt.show()
    print(f'✅  3D Preview rendered at frame {preview_fi}.')
else:
    print('⚠️  Could not read preview frame; continuing anyway.')

print(f'\n   Tube radius       : {TUBE_RADIUS} px')
print(f'   Trail length      : {TRAIL_MAX_FRAMES} frames')


✅  3D Preview rendered at frame 13.

   Tube radius       : 8 px
   Trail length      : 72 frames


In [15]:
# ─────────────────────────────────────────────────────────────
# Cell 11 · Write output_drs.mp4  (3D trajectory)
# ─────────────────────────────────────────────────────────────
OUTPUT_PATH = 'output_drs.mp4'

fourcc = cv2.VideoWriter_fourcc(*'avc1')
writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, FPS,
                         (FRAME_WIDTH, FRAME_HEIGHT))
if not writer.isOpened():
    print('avc1 not available, trying mp4v …')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, FPS,
                             (FRAME_WIDTH, FRAME_HEIGHT))

if not writer.isOpened():
    raise RuntimeError('Cannot open VideoWriter. Check codec support.')

print(f'🎬  Writing {OUTPUT_PATH}  ({FRAME_WIDTH}×{FRAME_HEIGHT} @ {FPS:.1f} fps) …')

cap       = cv2.VideoCapture(VIDEO_PATH)
frame_idx = 0
written   = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    out_frame = render_drs_frame(
        frame,
        frame_idx,
        smooth_x, smooth_y, smooth_z,
        radius=TUBE_RADIUS,
        trail_length=TRAIL_MAX_FRAMES
    )

    writer.write(out_frame)
    written   += 1
    frame_idx += 1

    if frame_idx % 50 == 0:
        pct = frame_idx / max(ACTUAL_FRAMES, 1) * 100
        print(f'   Rendered {frame_idx}/{ACTUAL_FRAMES}  ({pct:.0f}%)')

cap.release()
writer.release()

size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f'\n✅  Output video written.')
print(f'   Path     : {OUTPUT_PATH}')
print(f'   Frames   : {written}')
print(f'   Size     : {size_mb:.2f} MB')

# ── Re-encode with ffmpeg ─────────────────────────────────────
import subprocess
ffmpeg_out = 'output_drs_final.mp4'
cmd = (
    f'ffmpeg -y -i {OUTPUT_PATH} '
    f'-vcodec libx264 -crf 23 -preset fast '
    f'-pix_fmt yuv420p {ffmpeg_out} '
    f'2>&1 | tail -5'
)
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
if os.path.exists(ffmpeg_out) and os.path.getsize(ffmpeg_out) > 1000:
    OUTPUT_FINAL = ffmpeg_out
    print(f'\n✅  ffmpeg re-encode successful → {ffmpeg_out}  '
          f'({os.path.getsize(ffmpeg_out)/1e6:.2f} MB)')
else:
    OUTPUT_FINAL = OUTPUT_PATH
    print('⚠️  ffmpeg not available or failed; using original mp4.')
    print(result.stdout[-500:])


avc1 not available, trying mp4v …
🎬  Writing output_drs.mp4  (1920×1080 @ 30.4 fps) …
   Rendered 50/72  (69%)

✅  Output video written.
   Path     : output_drs.mp4
   Frames   : 72
   Size     : 1.57 MB

✅  ffmpeg re-encode successful → output_drs_final.mp4  (0.99 MB)


In [16]:
# ─────────────────────────────────────────────────────────────
# Cell 12 · Download output video to local computer
# ─────────────────────────────────────────────────────────────
from google.colab import files

if not os.path.exists(OUTPUT_FINAL):
    raise FileNotFoundError(f'{OUTPUT_FINAL} not found. Re-run Cell 11.')

print(f'⬇️  Downloading  {OUTPUT_FINAL}  ({os.path.getsize(OUTPUT_FINAL)/1e6:.2f} MB) …')
files.download(OUTPUT_FINAL)
print('✅  Download triggered.  Check your browser downloads.')

⬇️  Downloading  output_drs_final.mp4  (0.99 MB) …


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅  Download triggered.  Check your browser downloads.
